# Road-Safety Hotspot Analysis — ArcGIS Pro / `arcpy`

Turns NZTA Crash Analysis System (CAS) points into statistically significant crash
hotspots, discrete black-spot clusters, and a space-time trend classification for
Canterbury (full history).

**Run on the ArcGIS Pro Python kernel** (`arcgispro-py3` or a clone) — `arcpy` needs a
licensed ArcGIS Pro install. **Kernel Density** needs the **Spatial Analyst** extension;
the Hot Spot, Clustering and Space-Time tools are core.

**Input:** `CAS_Canterbury.geojson` — Canterbury only, 2000–present, downloaded from the
NZTA open data portal. The working layer is named **`crashes_canterbury`**.

> **Data note:** open CAS is **annual** (`crashYear`) — no month/day/time, so time
> analysis works in yearly steps. Public coordinates are snapped/offset for privacy, so
> treat hotspot *areas*, not exact points, as the signal.

> **Lock tip:** if a re-import ever fails with "schema lock / already exists", change
> the `RAW` name in Step 1 to a fresh value (or close the `.aprx` / collapse the GDB in
> Pro's Catalog pane to release the lock).

In [ ]:
# --- Environment + arcpy sanity check ---------------------------------------
import sys
try:
    import arcpy
except ModuleNotFoundError:
    raise SystemExit(
        "arcpy not found — wrong kernel. Select the ArcGIS Pro Python environment "
        "(arcgispro-py3 or a clone), then re-run."
    )
import os

print("Python:", sys.executable)
print("ArcGIS Pro version:", arcpy.GetInstallInfo()["Version"])

# --- Project paths ----------------------------------------------------------
PROJECT_DIR = r"C:\GISProjects\Road-safety hotspot analysis"
PROJECT_GDB = os.path.join(PROJECT_DIR, "Road-Safety Hotspot Analysis.gdb")

arcpy.env.workspace = PROJECT_GDB
arcpy.env.overwriteOutput = True

NZTM = arcpy.SpatialReference(2193)   # NZ Transverse Mercator 2000
arcpy.env.outputCoordinateSystem = NZTM

print("GDB exists:", arcpy.Exists(PROJECT_GDB))

In [ ]:
# --- CONFIG (confirmed against the CAS data) --------------------------------
CONFIG = {
    "severity_field": "crashSeverity",
    "year_field":     "crashYear",
    "region_field":   "region",
    "fatal_field":    "fatalCount",
    "serious_field":  "seriousInjuryCount",
    "minor_field":    "minorInjuryCount",

    "region_value":   "Canterbury Region",

    "severity_weights": {
        "Fatal Crash": 8,
        "Serious Crash": 4,
        "Minor Crash": 2,
        "Non-Injury Crash": 1,
    },
}

RAW = "crashes_canterbury"   # imported points (native CRS)
crashes = "crashes_nztm"     # final working layer: study-area subset in NZTM

## Step 1 — Import the GeoJSON (Z-safe, fresh output name)

The CAS GeoJSON carries Z-aware geometry; importing it while the output CRS env is a
projected system (2193) with no Z domain throws `ERROR 160660`. Fix: disable Z output
and import in native WGS84, then project in Step 1c. Writing to a fresh `RAW` name also
avoids the schema-lock issue on re-runs. (~132 MB → allow a couple of minutes.)

In [ ]:
# --- Step 1 -----------------------------------------------------------------
INPUT_GEOJSON = os.path.join(PROJECT_DIR, "CAS_Canterbury.geojson")

_saved_sr = arcpy.env.outputCoordinateSystem
_saved_z  = arcpy.env.outputZFlag
arcpy.env.outputCoordinateSystem = None       # import in native WGS84
arcpy.env.outputZFlag = "Disabled"            # drop Z -> no Z-domain requirement
try:
    arcpy.conversion.JSONToFeatures(INPUT_GEOJSON, RAW, "POINT")
finally:
    arcpy.env.outputCoordinateSystem = _saved_sr
    arcpy.env.outputZFlag = _saved_z

print(arcpy.management.GetCount(RAW)[0], "crashes imported")

## Step 1b — Inspect scope (fields, regions, severities, year span)

Confirms the data matches `CONFIG` and reports the **year span** — which decides whether
the space-time step (Step 5) is viable.

In [ ]:
# --- Step 1b ----------------------------------------------------------------
def distinct(fc, field):
    with arcpy.da.SearchCursor(fc, [field]) as cur:
        return sorted({v for (v,) in cur if v is not None})

print("FIELDS:")
print([f.name for f in arcpy.ListFields(RAW)])
print("\nREGION VALUES:", distinct(RAW, CONFIG["region_field"]))
print("\nSEVERITY VALUES:", distinct(RAW, CONFIG["severity_field"]))

yrs = [int(y) for y in distinct(RAW, CONFIG["year_field"])]
print(f"\nYEAR RANGE: {min(yrs)}-{max(yrs)}  ({len(yrs)} distinct years)")

## Step 1c — Filter to Canterbury, project to NZTM

The download is already Canterbury-only, so this filter mainly guards against any stray
rows and projects everything to NZTM for analysis.

In [ ]:
# --- Step 1c ----------------------------------------------------------------
where = f"{CONFIG['region_field']} = '{CONFIG['region_value']}'"
arcpy.conversion.ExportFeatures(RAW, "crashes_subset", where_clause=where)
arcpy.management.Project("crashes_subset", crashes, NZTM)

n = int(arcpy.management.GetCount(crashes)[0])
print(n, "crashes in study area")
if n == 0:
    print("*** 0 rows — check region_value against the Step 1b output. ***")

## Step 1d — Derived fields

A numeric **severity weight** (serious crashes count for more in the density surface)
and a **DATE** built from `crashYear` (annual → 1 July each year) for the space-time cube.

In [ ]:
# --- Step 1d ----------------------------------------------------------------
arcpy.management.AddField(crashes, "sevWeight", "SHORT")
_cb = f"""
weights = {CONFIG['severity_weights']}
def wt(sev):
    return weights.get(sev, 1)
"""
arcpy.management.CalculateField(
    crashes, "sevWeight", f"wt(!{CONFIG['severity_field']}!)",
    "PYTHON3", code_block=_cb,
)

arcpy.management.AddField(crashes, "crashDate", "DATE")
arcpy.management.CalculateField(
    crashes, "crashDate",
    f"datetime.datetime(int(!{CONFIG['year_field']}!), 7, 1)",
    "PYTHON3", code_block="import datetime",
)
print("Derived fields added: sevWeight, crashDate")

## Step 2 — Kernel Density (Spatial Analyst)

A continuous, severity-weighted density surface for the overview map. Cell size and
search radius are in metres (NZTM units) — tune to your study area.

In [ ]:
# --- Step 2 -----------------------------------------------------------------
arcpy.CheckOutExtension("Spatial")
kd = arcpy.sa.KernelDensity(
    in_features=crashes,
    population_field="sevWeight",       # "NONE" for raw counts
    cell_size=50,                        # metres
    search_radius=500,                   # metres
    area_unit_scale_factor="SQUARE_KILOMETERS",
)
kd.save(os.path.join(PROJECT_GDB, "crash_density"))
arcpy.CheckInExtension("Spatial")
print("Kernel density surface written: crash_density")

## Step 3 — Optimized Hot Spot Analysis (Getis-Ord Gi\*)

Counts incidents in a fishnet, picks a distance band, and returns **statistically
significant** hot/cold spots with confidence levels — real clusters, not just dense ones.

In [ ]:
# --- Step 3 -----------------------------------------------------------------
arcpy.stats.OptimizedHotSpotAnalysis(
    Input_Features=crashes,
    Output_Features=os.path.join(PROJECT_GDB, "crash_hotspots"),
    Analysis_Field=None,
    Incident_Data_Aggregation_Method="COUNT_INCIDENTS_WITHIN_FISHNET_POLYGONS",
)
print("Hot spot analysis written: crash_hotspots")

## Step 4 — Density-based clustering (DBSCAN)

Groups crashes into discrete black-spots you can label, count and rank into a
priority-sites table.

In [ ]:
# --- Step 4 -----------------------------------------------------------------
arcpy.stats.DensityBasedClustering(
    in_features=crashes,
    output_features=os.path.join(PROJECT_GDB, "crash_clusters"),
    cluster_method="DBSCAN",
    min_features_cluster=10,
    search_distance="150 Meters",
)
print("Clusters written: crash_clusters")

## Step 5 — Space-Time Cube + Emerging Hot Spot Analysis

The showpiece: classifies each location's trend over time (new, intensifying, persistent,
diminishing, sporadic, oscillating…). Needs several years — the guard below checks the
span and skips gracefully if there aren't enough.

In [ ]:
# --- Step 5 -----------------------------------------------------------------
yrs = {int(y) for y in
       (v for (v,) in arcpy.da.SearchCursor(crashes, [CONFIG["year_field"]]) if v is not None)}
n_years = len(yrs)
print(f"Distinct years in study area: {n_years}"
      + (f"  ({min(yrs)}-{max(yrs)})" if yrs else ""))

MIN_YEARS = 3
if n_years < MIN_YEARS:
    print("\n*** Skipping Emerging Hot Spot Analysis — not enough time steps. ***")
    print("Re-download CAS across a multi-year span (ideally 10+ years) to run this step.")
else:
    cube = os.path.join(PROJECT_DIR, "crash_cube.nc")
    arcpy.stpm.CreateSpaceTimeCubeByAggregatingPoints(
        in_features=crashes,
        output_cube=cube,
        time_field="crashDate",
        time_step_interval="1 Years",
        distance_interval="250 Meters",
        aggregation_shape_type="FISHNET_GRID",
    )
    arcpy.stpm.EmergingHotSpotAnalysis(
        in_cube=cube,
        analysis_variable="COUNT",
        output_features=os.path.join(PROJECT_GDB, "crash_emerging_hotspots"),
        neighborhood_distance="500 Meters",
        neighborhood_time_step=1,
    )
    print("Emerging hot spot analysis written: crash_emerging_hotspots")

## Step 6 — Summary tables for the write-up

Crashes per year (the trend) and by severity.

In [ ]:
# --- Step 6 -----------------------------------------------------------------
arcpy.analysis.Statistics(
    crashes, os.path.join(PROJECT_GDB, "summary_by_year"),
    [["OBJECTID", "COUNT"]], case_field=CONFIG["year_field"],
)
arcpy.analysis.Statistics(
    crashes, os.path.join(PROJECT_GDB, "summary_by_severity"),
    [["OBJECTID", "COUNT"],
     [CONFIG["fatal_field"], "SUM"],
     [CONFIG["serious_field"], "SUM"]],
    case_field=CONFIG["severity_field"],
)

import collections
by_year = collections.Counter()
with arcpy.da.SearchCursor(crashes, [CONFIG["year_field"]]) as cur:
    for (yr,) in cur:
        by_year[yr] += 1
for yr in sorted(k for k in by_year if k is not None):
    print(yr, by_year[yr])

## Step 7 — Share & document

**Publish the live app** (Pro GUI or the ArcGIS API for Python): share `crash_hotspots`
and `crash_emerging_hotspots` as a **hosted feature layer** to ArcGIS Online, build a
**Dashboard** or **Instant App**, and put the live URL at the top of your README — that
also ticks the web-GIS box in the job description.

**Commit to GitHub:** this notebook, exported map layouts (`docs/*.png|pdf`), and a
`README.md` (problem → data → method → results → limitations, maps embedded, live link up
top). **Don't commit** the raw `.geojson` (link + attribute — CAS is CC BY 4.0) or the
`.aprx` / `.nc` binaries.

## Limitations & honest notes (adapt into your README)

- **Annual granularity.** Open CAS has `crashYear` only — no time-of-day / day-of-week.
- **Location offset.** Public CAS coordinates are snapped/offset for privacy — hotspot
  *areas* are the reliable signal, not exact positions.
- **Reported crashes only.** Under-reporting is greatest for minor / non-injury crashes;
  recent years may also be incomplete due to reporting/processing lag.
- **Parameter tuning.** Cell sizes, radii, `min_features_cluster` and cube intervals
  should be justified for the study area, not left at these defaults.